In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.ensemble import RandomForestClassifier
import joblib
import os
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report


In [2]:
df=pd.read_excel('../data/enriched_data.xlsx')

In [3]:
df.head()

,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips,ip,...,ua_browser_version,ua_os,ua_os_version,ua_is_mobile,ua_is_pc,country,city,region,timezone,org
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,['185.97.201.89'],185.97.201.89,...,78.0.3904,Mac OS X,10.15.1,False,True,Russia,Saint Petersburg,St.-Petersburg,Europe/Moscow,AS39087 P.A.K.T LLC
1,02591b70-e8f5-4299-901c-e78b2b79b526,unknown,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"['64.52.83.86', '154.160.9.113', '64.52.83.10']",64.52.83.86,...,12.1.2,iOS,12.4.1,True,False,United States,San Jose,California,America/Los_Angeles,AS12182 Unitas Global
2,02591b70-e8f5-4299-901c-e78b2b79b526,unknown,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"['64.52.83.86', '154.160.9.113', '64.52.83.10']",154.160.9.113,...,12.1.2,iOS,12.4.1,True,False,Ghana,Ashaiman,Greater Accra,Africa/Accra,AS30986 Scancom Limited
3,02591b70-e8f5-4299-901c-e78b2b79b526,unknown,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"['64.52.83.86', '154.160.9.113', '64.52.83.10']",64.52.83.10,...,12.1.2,iOS,12.4.1,True,False,United States,San Jose,California,America/Los_Angeles,AS12182 Unitas Global
4,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,['154.160.10.201'],154.160.10.201,...,12.1.2,iOS,12.4.1,True,False,Ghana,Accra,Greater Accra,Africa/Accra,AS30986 Scancom Limited


## Data Preparation

In [4]:
compromised_device_id = "91b12379-8098-457f-a2ad-a94d767797c2"
compromised_identity = "0007f265568f1abc1da791e852877df2047b3af9"

In [5]:
# Identify connections directly linked to compromised identifiers
df["is_compromised_device_id"] = (df["device_id"] == compromised_device_id).astype(int)
df["is_compromised_identity"] = (df["identity"] == compromised_identity).astype(int)

In [6]:
df["is_compromised_device_id"].value_counts()

is_compromised_device_id
0    1799
1       3
Name: count, dtype: int64

In [7]:
df["is_compromised_identity"].value_counts()

is_compromised_identity
0    1726
1      76
Name: count, dtype: int64

In [8]:
# Create the primary fraud label based on direct compromise
df["fraud_label"] = ((df["is_compromised_device_id"] == 1) | (df["is_compromised_identity"] == 1)).astype(int)

In [9]:
# Identify IPs and Fingerprints associated with the directly compromised connections
compromised_connections = df[df["fraud_label"] == 1]
related_ips = set(compromised_connections["ip"].unique())
related_fingerprints = set(compromised_connections["device_fingerprint"].unique())

In [10]:
len(related_ips), len(related_fingerprints)

(72, 2)

In [11]:
# Create features indicating sharing of related identifiers
df["shares_compromised_ip"] = df["ip"].apply(lambda x: 1 if x in related_ips else 0)
df["shares_compromised_fingerprint"] = df["device_fingerprint"].apply(lambda x: 1 if x in related_fingerprints else 0)


In [12]:
df.head()

,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips,ip,...,country,city,region,timezone,org,is_compromised_device_id,is_compromised_identity,fraud_label,shares_compromised_ip,shares_compromised_fingerprint
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,['185.97.201.89'],185.97.201.89,...,Russia,Saint Petersburg,St.-Petersburg,Europe/Moscow,AS39087 P.A.K.T LLC,0,0,0,0,0
1,02591b70-e8f5-4299-901c-e78b2b79b526,unknown,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"['64.52.83.86', '154.160.9.113', '64.52.83.10']",64.52.83.86,...,United States,San Jose,California,America/Los_Angeles,AS12182 Unitas Global,0,0,0,0,0
2,02591b70-e8f5-4299-901c-e78b2b79b526,unknown,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"['64.52.83.86', '154.160.9.113', '64.52.83.10']",154.160.9.113,...,Ghana,Ashaiman,Greater Accra,Africa/Accra,AS30986 Scancom Limited,0,0,0,0,0
3,02591b70-e8f5-4299-901c-e78b2b79b526,unknown,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"['64.52.83.86', '154.160.9.113', '64.52.83.10']",64.52.83.10,...,United States,San Jose,California,America/Los_Angeles,AS12182 Unitas Global,0,0,0,0,0
4,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,['154.160.10.201'],154.160.10.201,...,Ghana,Accra,Greater Accra,Africa/Accra,AS30986 Scancom Limited,0,0,0,0,0


In [13]:
#replace all - to UNKNOWN
df = df.replace("-", "UNKNOWN")

In [14]:
df.head()

,device_id,identity,bank,device_fingerprint,gpu_renderers,screen,os,browser,ips,ip,...,country,city,region,timezone,org,is_compromised_device_id,is_compromised_identity,fraud_label,shares_compromised_ip,shares_compromised_fingerprint
0,00993a56-f44b-448d-9d33-78c16f0be636,37294473,Bank1,b48af28979-116c56e539-9b9e431e93-87a831f69f,Intel(R) Iris(TM) Plus Graphics 655,"(1440, 900, 24)",Mac OS X 10.15.1,Chrome 78.0.3904,['185.97.201.89'],185.97.201.89,...,Russia,Saint Petersburg,St.-Petersburg,Europe/Moscow,AS39087 P.A.K.T LLC,0,0,0,0,0
1,02591b70-e8f5-4299-901c-e78b2b79b526,unknown,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"['64.52.83.86', '154.160.9.113', '64.52.83.10']",64.52.83.86,...,United States,San Jose,California,America/Los_Angeles,AS12182 Unitas Global,0,0,0,0,0
2,02591b70-e8f5-4299-901c-e78b2b79b526,unknown,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"['64.52.83.86', '154.160.9.113', '64.52.83.10']",154.160.9.113,...,Ghana,Ashaiman,Greater Accra,Africa/Accra,AS30986 Scancom Limited,0,0,0,0,0
3,02591b70-e8f5-4299-901c-e78b2b79b526,unknown,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,"['64.52.83.86', '154.160.9.113', '64.52.83.10']",64.52.83.10,...,United States,San Jose,California,America/Los_Angeles,AS12182 Unitas Global,0,0,0,0,0
4,02591b70-e8f5-4299-901c-e78b2b79b526,2577384,Bank2,6413e837d3-987e3415d7-902dcc5844-27c94f86d9,Apple GPU,"(896, 414, 32)",iOS 12.4.1,Mobile Safari 12.1.2,['154.160.10.201'],154.160.10.201,...,Ghana,Accra,Greater Accra,Africa/Accra,AS30986 Scancom Limited,0,0,0,0,0


## Feature Selection & Encoding


In [15]:
# Select features for the model
categorical_features = ["bank", "os", "browser", "country", "city", "region", "timezone", "org"]
boolean_features = ["ua_is_mobile", "ua_is_pc"]
engineered_features = ["shares_compromised_ip", "shares_compromised_fingerprint"]
# We won't use is_compromised_device_id/identity directly as features to avoid data leakage, 
# but they were crucial for labeling and finding related IPs/fingerprints.

In [16]:

features_to_use = categorical_features + boolean_features + engineered_features
target = "fraud_label"

In [17]:

df_model = df[features_to_use + [target]].copy()



In [18]:
# Convert boolean features to integers (True=1, False=0)
for col in boolean_features:
    df_model[col] = df_model[col].astype(int)



In [19]:
# Apply Label Encoding to categorical features (simple approach for now)
# Note: One-Hot Encoding might be better but increases dimensionality significantly.
# Label Encoding is simpler for a basic model but implies ordinal relationships which might not exist.
encoders = {}
for col in categorical_features:
    le = LabelEncoder()
    df_model[col] = le.fit_transform(df_model[col].astype(str)) # Handle potential non-string types
    encoders[col] = le # Store encoders if needed later for new data



In [20]:
# --- Train/Test Split ---
X = df_model[features_to_use]
y = df_model[target]

# Split data, ensuring stratification to handle potential class imbalance
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## Modeling

In [21]:

# Initialize a simple RandomForestClassifier model
# Using class_weight=\"balanced\" to handle potential imbalance from fraud labels
model = RandomForestClassifier(n_estimators=100, random_state=42, class_weight="balanced", n_jobs=-1)

# Train the model
model.fit(X_train, y_train)

print("Model training completed.")



Model training completed.


## Evaluate the model on the test set

In [22]:
# Evaluate the model
from sklearn.metrics import classification_report, confusion_matrix
y_pred = model.predict(X_test)
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))


              precision    recall  f1-score   support

           0       1.00      0.99      1.00       345
           1       0.89      1.00      0.94        16

    accuracy                           0.99       361
   macro avg       0.94      1.00      0.97       361
weighted avg       1.00      0.99      0.99       361

[[343   2]
 [  0  16]]


In [23]:
# Calculate metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, zero_division=0) # Handle cases with no predicted positives
recall = recall_score(y_test, y_pred, zero_division=0)       # Handle cases with no actual positives
f1 = f1_score(y_test, y_pred, zero_division=0)             # Handle cases with no positives
conf_matrix = confusion_matrix(y_test, y_pred)
class_report = classification_report(y_test, y_pred, zero_division=0, target_names=["Non-Fraud (0)", "Fraud (1)"])


In [24]:
# Prepare results string
results_str = f"Model Evaluation Results:\n"
results_str += f"=========================\n"
results_str += f"Test Set Size: {len(X_test)} samples\n"
results_str += f"Accuracy: {accuracy:.4f}\n"
results_str += f"Precision (for Fraud class 1): {precision:.4f}\n"
results_str += f"Recall (for Fraud class 1): {recall:.4f}\n"
results_str += f"F1-Score (for Fraud class 1): {f1:.4f}\n\n"
results_str += f"Confusion Matrix:\n"
results_str += f"[[TN FP] [FN TP]]\n"
results_str += f"{conf_matrix}\n\n"
results_str += f"Classification Report:\n"
results_str += f"{class_report}\n"

# Print results to console
print(results_str)

Model Evaluation Results:
Test Set Size: 361 samples
Accuracy: 0.9945
Precision (for Fraud class 1): 0.8889
Recall (for Fraud class 1): 1.0000
F1-Score (for Fraud class 1): 0.9412

Confusion Matrix:
[[TN FP] [FN TP]]
[[343   2]
 [  0  16]]

Classification Report:
               precision    recall  f1-score   support

Non-Fraud (0)       1.00      0.99      1.00       345
    Fraud (1)       0.89      1.00      0.94        16

     accuracy                           0.99       361
    macro avg       0.94      1.00      0.97       361
 weighted avg       1.00      0.99      0.99       361




In [25]:
# Make predictions on the entire dataset
predictions = model.predict(X)
probabilities = model.predict_proba(X)[:, 1] # Probability of being fraud (class 1)

In [26]:
# Use original_df index to ensure alignment
df["predicted_fraud_label"] = predictions
df["predicted_fraud_probability"] = probabilities

In [27]:
# Save the full original data with predictions
df.to_csv('all_enriched_data_fraud_labeled.csv', index=False)
print(f"Saved all connections with predictions to: {"all_enriched_data_fraud_labeled.csv"}")

Saved all connections with predictions to: all_enriched_data_fraud_labeled.csv


In [28]:
fraudulent_connections_df = df[df["predicted_fraud_label"] == 1].copy()
fraudulent_connections_df = fraudulent_connections_df.sort_values(by="predicted_fraud_probability", ascending=False)
fraudulent_connections_df.to_csv("fraudlent_connection.csv", index=False)
print(f"Saved top fraudulent connections to: {"fraudlent_connection.csv"}")


Saved top fraudulent connections to: fraudlent_connection.csv


## THE END   